In [1]:
from pathlib import Path
import pandas as pd
import sqlite3

RAW_DATA_DIR = Path("../data/raw")

conn = sqlite3.connect("../data/olist.db")

In [2]:
orders = pd.read_csv(
    RAW_DATA_DIR / "olist_orders_dataset.csv"
)

reviews = pd.read_csv(
    RAW_DATA_DIR / "olist_order_reviews_dataset.csv"
)

orders.to_sql(
    "olist_orders_dataset",
    conn,
    if_exists="replace",
    index=False
)

reviews.to_sql(
    "olist_order_reviews_dataset",
    conn,
    if_exists="replace",
    index=False
)

99224

In [5]:
query = """
WITH review_by_order AS (
    SELECT
        order_id,
        AVG(review_score) AS review_score
    FROM olist_order_reviews_dataset
    GROUP BY order_id
),

delivery_performance AS (
    SELECT
        order_id,
        CASE
            WHEN date(order_delivered_customer_date)
                 > date(order_estimated_delivery_date)
            THEN 1
            ELSE 0
        END AS is_late
    FROM olist_orders_dataset
    WHERE order_delivered_customer_date IS NOT NULL
)

SELECT
    d.is_late,
    COUNT(*) AS order_count,
    AVG(r.review_score) AS average_review_score
FROM delivery_performance AS d
LEFT JOIN review_by_order AS r
    ON d.order_id = r.order_id
GROUP BY d.is_late
ORDER BY d.is_late;
"""

result = pd.read_sql_query(query, conn)
result

,is_late,order_count,average_review_score
0,0,89941,4.290450
1,1,6535,2.271937


In [6]:
query_2 = """
WITH delivery_performance AS (
    SELECT
        order_id,
        CASE
            WHEN date(order_delivered_customer_date)
                 > date(order_estimated_delivery_date)
            THEN 1
            ELSE 0
        END AS is_late,

        CAST(
            julianday(order_delivered_carrier_date)
            - julianday(order_approved_at)
            AS INTEGER
        ) AS handling_days

    FROM olist_orders_dataset
    WHERE order_delivered_customer_date IS NOT NULL
      AND order_delivered_carrier_date IS NOT NULL
      AND order_approved_at IS NOT NULL
),

valid_orders AS (
    SELECT
        order_id,
        is_late,
        handling_days,
        CASE
            WHEN handling_days <= 1 THEN '0–1 days'
            WHEN handling_days <= 3 THEN '2–3 days'
            WHEN handling_days <= 7 THEN '4–7 days'
            ELSE '8+ days'
        END AS handling_group
    FROM delivery_performance
    WHERE handling_days >= 0
)

SELECT
    handling_group,
    COUNT(*) AS order_count,
    ROUND(AVG(is_late) * 100, 2) AS late_delivery_rate_pct
FROM valid_orders
GROUP BY handling_group
ORDER BY
    CASE handling_group
        WHEN '0–1 days' THEN 1
        WHEN '2–3 days' THEN 2
        WHEN '4–7 days' THEN 3
        WHEN '8+ days' THEN 4
    END;
"""

result_2 = pd.read_sql_query(query_2, conn)
result_2

,handling_group,order_count,late_delivery_rate_pct
0,0–1 days,51522,4.60
1,2–3 days,24237,6.31
2,4–7 days,15299,9.19
3,8+ days,4950,24.69


In [7]:
# Which sellers produce the largest number of late orders?
order_items = pd.read_csv(
    RAW_DATA_DIR / "olist_order_items_dataset.csv"
)

order_items.to_sql(
    "olist_order_items_dataset",
    conn,
    if_exists="replace",
    index=False
)

112650

In [8]:
query_3 = """
WITH delivery_performance AS (
    SELECT
        order_id,
        CASE
            WHEN date(order_delivered_customer_date)
                 > date(order_estimated_delivery_date)
            THEN 1
            ELSE 0
        END AS is_late,

        CAST(
            julianday(order_delivered_carrier_date)
            - julianday(order_approved_at)
            AS INTEGER
        ) AS handling_days

    FROM olist_orders_dataset
    WHERE order_delivered_customer_date IS NOT NULL
),

seller_orders AS (
    SELECT DISTINCT
        i.order_id,
        i.seller_id,
        d.is_late,
        d.handling_days
    FROM olist_order_items_dataset AS i
    INNER JOIN delivery_performance AS d
        ON i.order_id = d.order_id
),

seller_performance AS (
    SELECT
        seller_id,
        COUNT(DISTINCT order_id) AS order_count,
        SUM(is_late) AS late_order_count,
        AVG(is_late) * 100 AS late_delivery_rate_pct,
        AVG(handling_days) AS average_handling_days
    FROM seller_orders
    GROUP BY seller_id
)

SELECT
    seller_id,
    order_count,
    late_order_count,
    ROUND(late_delivery_rate_pct, 2) AS late_delivery_rate_pct,
    ROUND(average_handling_days, 2) AS average_handling_days
FROM seller_performance
WHERE order_count >= 30
ORDER BY late_order_count DESC
LIMIT 10;
"""

result_3 = pd.read_sql_query(query_3, conn)
result_3

,seller_id,order_count,late_order_count,late_delivery_rate_pct,average_handling_days
0,4a3ca9315b744ce9f8e9374361493884,1772,172,9.71,1.86
1,1f50f920176fa81dab994f9023523100,1399,124,8.86,2.94
2,4869f7a5dfa277a7dca6462dcf3b52b2,1124,118,10.50,1.83
3,6560211a19b47992c3666cc44a7e94c0,1819,96,5.28,0.71
4,ea8482cd71df3c1969d7b9473ff13abc,1132,96,8.48,2.40
5,7c67e1448b00f6e969d365cea6b010ab,973,89,9.15,10.86
6,cc419e0650a3c5ba77189a1882b7556a,1651,87,5.27,1.87
7,da8622b14eb17ae2831f4ac5b9dab84a,1311,87,6.64,1.74
8,8b321bb669392f5163d04c59e235e066,930,81,8.71,1.53
9,955fee9216a65b617aa5c0531780ce60,1261,77,6.11,1.22


In [9]:
# Which product categories combine high order volume and high late-order impact?
products = pd.read_csv(
    RAW_DATA_DIR / "olist_products_dataset.csv"
)

category_translation = pd.read_csv(
    RAW_DATA_DIR / "product_category_name_translation.csv"
)

products.to_sql(
    "olist_products_dataset",
    conn,
    if_exists="replace",
    index=False
)

category_translation.to_sql(
    "product_category_name_translation",
    conn,
    if_exists="replace",
    index=False
)

71

In [10]:
query_4 = """
WITH delivery_performance AS (
    SELECT
        order_id,
        CASE
            WHEN date(order_delivered_customer_date)
                 > date(order_estimated_delivery_date)
            THEN 1
            ELSE 0
        END AS is_late
    FROM olist_orders_dataset
    WHERE order_delivered_customer_date IS NOT NULL
),

order_categories AS (
    SELECT DISTINCT
        i.order_id,
        COALESCE(
            t.product_category_name_english,
            p.product_category_name
        ) AS product_category_name_english
    FROM olist_order_items_dataset AS i
    INNER JOIN olist_products_dataset AS p
        ON i.product_id = p.product_id
    LEFT JOIN product_category_name_translation AS t
        ON p.product_category_name = t.product_category_name
),

category_performance AS (
    SELECT
        c.product_category_name_english,
        COUNT(DISTINCT c.order_id) AS order_count,
        SUM(d.is_late) AS late_order_count,
        AVG(d.is_late) * 100 AS late_delivery_rate_pct
    FROM order_categories AS c
    INNER JOIN delivery_performance AS d
        ON c.order_id = d.order_id
    WHERE c.product_category_name_english IS NOT NULL
    GROUP BY c.product_category_name_english
)

SELECT
    product_category_name_english,
    order_count,
    late_order_count,
    ROUND(late_delivery_rate_pct, 2) AS late_delivery_rate_pct
FROM category_performance
WHERE order_count >= 100
ORDER BY late_order_count DESC
LIMIT 10;
"""

result_4 = pd.read_sql_query(query_4, conn)
result_4

,product_category_name_english,order_count,late_order_count,late_delivery_rate_pct
0,bed_bath_table,9272,689,7.43
1,health_beauty,8649,650,7.52
2,sports_leisure,7530,495,6.57
3,furniture_decor,6307,449,7.12
4,computers_accessories,6529,417,6.39
5,watches_gifts,5493,406,7.39
6,housewares,5743,308,5.36
7,telephony,4093,291,7.11
8,auto,3809,278,7.30
9,toys,3804,243,6.39
